# Section 8: Lifecycle & Flow Analysis (Q69–Q76)
State machine transitions, material/financial flow tracing, mass balance, and what-if scenarios.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import get_session, run_sql
import pandas as pd
conn, ontology = get_session()

## Q69
Find an order in "pending" status. What are the valid next states it can transition to? Show me the allowed moves from where it sits today.

In [ ]:
# Read state machine from ontology
sm = ontology.get_class_state_machine('Order')
print("Order state machine:")
for key, val in sm.items():
    print(f"  {key}: {val}")

# Find valid transitions from 'pending'
transitions = sm.get('transitions', [])
pending_transitions = [t for t in transitions if t.get('from') == 'pending']
print(f"\nValid transitions from 'pending': {[t.get('to') for t in pending_transitions]}")

# Find an actual pending order
print("\nExample pending order:")
run_sql(conn, """
    SELECT order_number, status, day, total_cases
    FROM orders
    WHERE status = 'pending'
    LIMIT 1
""")

## Q70
A warehouse manager tried to mark a shipment in "planned" status as "delivered". Is that a valid transition, or does it need to go through intermediate states first? What's the correct sequence?

In [ ]:
sm = ontology.get_class_state_machine('Shipment')
print("Shipment state machine:")
for key, val in sm.items():
    print(f"  {key}: {val}")

transitions = sm.get('transitions', [])
planned_to_delivered = [t for t in transitions if t.get('from') == 'planned' and t.get('to') == 'delivered']

if planned_to_delivered:
    print("\nplanned -> delivered IS a valid direct transition.")
else:
    print("\nplanned -> delivered is NOT a valid direct transition.")
    # Find the valid path
    from_planned = [t for t in transitions if t.get('from') == 'planned']
    print(f"From 'planned', valid transitions: {[t.get('to') for t in from_planned]}")
    # Show full path
    print("\nCorrect sequence: planned -> in_transit -> delivered")

## Q71
Trace the material flow for Glycerin USP (ACT-HUMECTANT-001): start from the purchase orders, through goods receipts, into production batches, and out through shipments. Show the quantity in kg at each stage so I can see where volume accumulates or drops.

In [ ]:
ingredient_code = 'ACT-HUMECTANT-001'

# Stage 1: Purchase Orders
po_qty = run_sql(conn, """
    SELECT 'Purchase Orders' as stage,
           COUNT(DISTINCT po.id) as doc_count,
           SUM(pol.quantity_kg) as total_kg
    FROM purchase_order_lines pol
    JOIN purchase_orders po ON pol.po_id = po.id
    JOIN ingredients i ON pol.ingredient_id = i.id
    WHERE i.ingredient_code = %s
""", (ingredient_code,))

# Stage 2: Goods Receipts
gr_qty = run_sql(conn, """
    SELECT 'Goods Receipts' as stage,
           COUNT(DISTINCT gr.id) as doc_count,
           SUM(grl.quantity_kg) as total_kg
    FROM goods_receipt_lines grl
    JOIN goods_receipts gr ON grl.gr_id = gr.id
    JOIN ingredients i ON grl.ingredient_id = i.id
    WHERE i.ingredient_code = %s
""", (ingredient_code,))

# Stage 3: Batch Consumption
batch_qty = run_sql(conn, """
    SELECT 'Batch Consumption' as stage,
           COUNT(DISTINCT bting.batch_id) as doc_count,
           SUM(bting.quantity_kg) as total_kg
    FROM batch_ingredients bting
    JOIN ingredients i ON bting.ingredient_id = i.id
    WHERE i.ingredient_code = %s
""", (ingredient_code,))

# Stage 4: Finished goods shipped (estimate via batches -> SKUs -> shipment lines)
ship_qty = run_sql(conn, """
    SELECT 'Shipments (output)' as stage,
           COUNT(DISTINCT sl.shipment_id) as doc_count,
           SUM(sl.weight_kg) as total_kg
    FROM batch_ingredients bting
    JOIN ingredients i ON bting.ingredient_id = i.id
    JOIN batches b ON bting.batch_id = b.id
    JOIN shipment_lines sl ON sl.sku_id = b.product_id
    WHERE i.ingredient_code = %s
      AND b.product_type = 'finished_good'
""", (ingredient_code,))

flow = pd.concat([po_qty, gr_qty, batch_qty, ship_qty], ignore_index=True)
print(f"Material flow for {ingredient_code}:")
display(flow)

## Q72
For supplier SUP-008 (PrimePack International), trace the full financial flow: purchase orders, the corresponding AP invoices, and the payments against those invoices. Flag any AP invoices where the amount doesn't match the PO — I want to see our three-way match failures.

In [ ]:
# Financial flow: PO -> AP Invoice -> Payment
run_sql(conn, """
    SELECT po.po_number,
           SUM(pol.quantity_kg * pol.unit_cost) as po_value,
           api.invoice_number,
           api.total_amount as invoice_amount,
           ABS(SUM(pol.quantity_kg * pol.unit_cost) - api.total_amount) as mismatch,
           CASE WHEN ABS(SUM(pol.quantity_kg * pol.unit_cost) - api.total_amount) > 1.0
                THEN 'MISMATCH' ELSE 'OK' END as match_status,
           pay.payment_date,
           pay.amount as payment_amount,
           pay.discount_amount
    FROM suppliers s
    JOIN purchase_orders po ON po.supplier_id = s.id
    JOIN purchase_order_lines pol ON pol.po_id = po.id
    LEFT JOIN goods_receipts gr ON gr.plant_id = po.plant_id
    LEFT JOIN ap_invoices api ON api.supplier_id = s.id AND api.gr_id = gr.id
    LEFT JOIN ap_payments pay ON pay.invoice_id = api.id
    WHERE s.supplier_code = 'SUP-008'
    GROUP BY po.po_number, api.invoice_number, api.total_amount, pay.payment_date, pay.amount, pay.discount_amount
    ORDER BY po.po_number
    LIMIT 30
""")

## Q73
For batch B-001-000021, verify mass balance: does the total weight of ingredients consumed equal the batch output quantity plus expected scrap? Show me the numbers — I want to know if we have an unexplained variance.

In [ ]:
batch_info = run_sql(conn, """
    SELECT b.batch_number, b.quantity_kg as output_kg, b.yield_percent,
           f.formula_code, f.batch_size_kg as formula_batch_size
    FROM batches b
    JOIN formulas f ON b.formula_id = f.id
    WHERE b.batch_number = 'B-001-000021'
""")
display(batch_info)

ingredients_consumed = run_sql(conn, """
    SELECT i.ingredient_code, i.name, bting.quantity_kg
    FROM batch_ingredients bting
    JOIN ingredients i ON bting.ingredient_id = i.id
    WHERE bting.batch_id = (SELECT id FROM batches WHERE batch_number = 'B-001-000021')
    ORDER BY bting.quantity_kg DESC
""")
display(ingredients_consumed)

total_input = ingredients_consumed['quantity_kg'].sum()
output = float(batch_info['output_kg'].iloc[0])
yield_pct = float(batch_info['yield_percent'].iloc[0])
expected_output = total_input * (yield_pct / 100.0)
variance = output - expected_output

print(f"\nTotal input: {total_input:.2f} kg")
print(f"Batch output: {output:.2f} kg")
print(f"Yield percent: {yield_pct:.1f}%")
print(f"Expected output (input × yield): {expected_output:.2f} kg")
print(f"Variance: {variance:.2f} kg ({100*variance/expected_output:.1f}%)")

## Q74
Verify that debits equal credits in the general ledger for day 200. Pull the total debit and credit amounts and show any imbalance. Also flag if you see any duplicate invoice postings — we've had "-DUP" entries slip through.

In [ ]:
# GL balance check for day 200
balance = run_sql(conn, """
    SELECT SUM(debit_amount) as total_debits,
           SUM(credit_amount) as total_credits,
           SUM(debit_amount) - SUM(credit_amount) as imbalance
    FROM gl_journal
    WHERE entry_date = 200
""")
print("GL Balance for day 200:")
display(balance)

# Check for -DUP entries
dups = run_sql(conn, """
    SELECT id, entry_date, account_code, debit_amount, credit_amount,
           reference_type, reference_id, description
    FROM gl_journal
    WHERE entry_date = 200
      AND description LIKE '%-DUP%'
    ORDER BY id
""")
print(f"\nDuplicate entries (-DUP): {len(dups)}")
if len(dups) > 0:
    display(dups)

## Q75
Find an order in "pending" status. Check the preconditions for allocation: is it in the right status, and is there sufficient inventory at the destination DC for the SKUs on the order?

In [ ]:
# Find a pending order
pending = run_sql(conn, """
    SELECT o.id, o.order_number, o.status, o.retail_location_id,
           rl.location_code as retail_location
    FROM orders o
    JOIN retail_locations rl ON o.retail_location_id = rl.id
    WHERE o.status = 'pending'
    LIMIT 1
""")
display(pending)

order_id = int(pending['id'].iloc[0])

# Check order lines
lines = run_sql(conn, """
    SELECT ol.line_number, s.sku_code, ol.quantity_cases
    FROM order_lines ol
    JOIN skus s ON ol.sku_id = s.id
    WHERE ol.order_id = %s
""", (order_id,))
print("\nOrder lines:")
display(lines)

# Check inventory at nearby DCs for these SKUs
sku_ids = run_sql(conn, "SELECT DISTINCT sku_id FROM order_lines WHERE order_id = %s", (order_id,))
ph = ','.join(['%s'] * len(sku_ids))
inv = run_sql(conn, f"""
    SELECT dc.dc_code, s.sku_code, inv.quantity_cases
    FROM inventory inv
    JOIN distribution_centers dc ON inv.location_id = dc.id
    JOIN skus s ON inv.sku_id = s.id
    WHERE inv.sku_id IN ({ph})
      AND inv.location_type IN ('rdc', 'customer_dc')
      AND inv.day = (SELECT MAX(day) FROM inventory)
      AND inv.quantity_cases > 0
    ORDER BY dc.dc_code, s.sku_code
""", list(sku_ids['sku_id']))
print("\nAvailable DC inventory for order SKUs:")
display(inv)

# Precondition check
print(f"\nStatus check: {'PASS' if pending['status'].iloc[0] == 'pending' else 'FAIL'} (status = {pending['status'].iloc[0]})")
print(f"Inventory check: {'PASS' if len(inv) > 0 else 'FAIL'} ({len(inv)} inventory records found)")

## Q76
What happens to our production plan if the Dallas plant's (PLANT-TX) daily capacity drops by 20%? Which work orders would be affected, and how much planned volume exceeds the new capacity ceiling?

In [ ]:
# Get PLANT-TX capacity
plant = run_sql(conn, """
    SELECT id, plant_code, name, capacity_tons_per_day
    FROM plants WHERE plant_code = 'PLANT-TX'
""")
display(plant)

capacity = float(plant['capacity_tons_per_day'].iloc[0])
new_capacity = capacity * 0.8
print(f"\nCurrent capacity: {capacity:.1f} tons/day")
print(f"Reduced capacity (-20%): {new_capacity:.1f} tons/day")

# Work orders at PLANT-TX
wo = run_sql(conn, """
    SELECT wo.wo_number, wo.status, wo.planned_quantity_kg,
           wo.planned_start_date, wo.due_date,
           f.formula_code
    FROM work_orders wo
    JOIN plants p ON wo.plant_id = p.id
    JOIN formulas f ON wo.formula_id = f.id
    WHERE p.plant_code = 'PLANT-TX'
      AND wo.status IN ('planned', 'in_progress')
    ORDER BY wo.planned_start_date
""")
print(f"\nActive/planned work orders at PLANT-TX: {len(wo)}")
display(wo)

# Aggregate daily demand
daily = run_sql(conn, """
    SELECT wo.planned_start_date as day,
           SUM(wo.planned_quantity_kg) / 1000.0 as planned_tons
    FROM work_orders wo
    JOIN plants p ON wo.plant_id = p.id
    WHERE p.plant_code = 'PLANT-TX'
      AND wo.status IN ('planned', 'in_progress')
    GROUP BY wo.planned_start_date
    HAVING SUM(wo.planned_quantity_kg) / 1000.0 > %s
    ORDER BY wo.planned_start_date
""", (new_capacity,))
print(f"\nDays exceeding new capacity ceiling ({new_capacity:.1f} tons): {len(daily)}")
if len(daily) > 0:
    display(daily)

In [ ]:
conn.close()
print("Session closed.")